In [68]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("credit_data_realistic_v2_dirty.csv")
df = df_raw.copy()


In [69]:
df.head()

,customer_id,age,gender,education_level,marital_status,employment_years,annual_income,monthly_income,credit_limit,used_credit,...,pay_amt_m4,pay_delay_m5,bill_amt_m5,pay_amt_m5,pay_delay_m6,bill_amt_m6,pay_amt_m6,past_defaults,num_late_payments,default_flag
0,1000000.0,150.0,female,Graduate,Married,18.0,396761.616683,33806.073284,403095.363533,193400.490816,...,30220.707995,0.0,115903.383155,51701.668950,0.0,82073.852822,114366.801393,0.0,9,1
1,1000001.0,49.0,M,Graduate,Divorced,27.0,496985.668167,46520.149916,379085.659400,40240.397130,...,67083.304001,0.0,60251.089694,79115.695858,0.0,115966.313982,105733.344703,0.0,0,0
2,1000002.0,200.0,M,High School,Single,24.0,175713.151313,16140.987870,222802.315890,108651.935608,...,16033.248503,2.0,78388.974744,38798.966736,4.0,0.000000,0.000000,1.0,6,1
3,1000003.0,63.0,Male,Graduate,Married,4.0,506620.706896,38233.150598,460208.373993,162978.574828,...,4910.703712,0.0,97549.920990,126281.117583,2.0,36885.153081,22755.141979,1.0,2,0
4,1000004.0,28.0,F,Post-Graduate,Divorced,27.0,NaN,15467.155318,108546.431626,48646.491464,...,0.000000,0.0,87313.245796,41638.570934,0.0,0.000000,0.000000,0.0,0,0


In [70]:
df.columns.tolist()


['customer_id',
 'age',
 'gender',
 'education_level',
 'marital_status',
 'employment_years',
 'annual_income',
 'monthly_income',
 'credit_limit',
 'used_credit',
 'loan_amount',
 'credit_utilization',
 'num_credit_cards',
 'num_active_loans',
 'pay_delay_m1',
 'bill_amt_m1',
 'pay_amt_m1',
 'pay_delay_m2',
 'bill_amt_m2',
 'pay_amt_m2',
 'pay_delay_m3',
 'bill_amt_m3',
 'pay_amt_m3',
 'pay_delay_m4',
 'bill_amt_m4',
 'pay_amt_m4',
 'pay_delay_m5',
 'bill_amt_m5',
 'pay_amt_m5',
 'pay_delay_m6',
 'bill_amt_m6',
 'pay_amt_m6',
 'past_defaults',
 'num_late_payments',
 'default_flag']

In [71]:
profile = pd.DataFrame({
    "data_type": df.dtypes,
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100,
    "unique_values": df.nunique(),
})

profile = profile.sort_values("missing_pct", ascending=False)
profile


,data_type,missing_count,missing_pct,unique_values
annual_income,float64,9538,9.538,87040
employment_years,float64,5842,5.842,40
education_level,object,5000,5.000,6
marital_status,object,5000,5.000,3
credit_utilization,float64,5000,5.000,95000
age,float64,4903,4.903,53
pay_delay_m2,float64,3000,3.000,9
pay_amt_m1,float64,3000,3.000,90293
bill_amt_m4,float64,3000,3.000,90314
pay_delay_m4,float64,3000,3.000,9


In [72]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns

In [73]:
df[num_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
customer_id,99000.0,1.050021e+06,2.886617e+04,1000000.000000,1.025021e+06,1.050022e+06,1.075016e+06,1.099999e+06
age,95097.0,4.584968e+01,1.997000e+01,-5.000000,3.300000e+01,4.500000e+01,5.700000e+01,2.000000e+02
employment_years,94158.0,1.910007e+01,1.174860e+01,0.000000,9.000000e+00,1.900000e+01,2.900000e+01,3.900000e+01
annual_income,90462.0,6.234279e+05,1.053090e+06,-50000.000000,3.127735e+05,4.468384e+05,6.427102e+05,5.000000e+07
monthly_income,97000.0,4.186494e+04,2.274724e+04,10627.696888,2.604662e+04,3.673303e+04,5.188212e+04,3.287118e+05
credit_limit,100000.0,3.681014e+05,2.289072e+05,35768.911097,1.960863e+05,3.097099e+05,4.806286e+05,1.000000e+06
used_credit,100000.0,2.182647e+05,2.032371e+05,0.000000,8.108603e+04,1.615677e+05,2.927011e+05,2.962160e+06
loan_amount,100000.0,2.952012e+05,2.800376e+05,20000.000000,1.231577e+05,2.177032e+05,3.776472e+05,1.147628e+07
credit_utilization,95000.0,5.998173e-01,2.884564e-01,0.100003,3.499547e-01,6.004580e-01,8.485430e-01,1.099983e+00
num_credit_cards,100000.0,1.996670e+00,1.412579e+00,0.000000,1.000000e+00,2.000000e+00,3.000000e+00,1.100000e+01


In [74]:
invalid_checks = {
    "age_invalid": ((df["age"] < 18) | (df["age"] > 100)).sum(),
    "income_invalid": (df["annual_income"] <= 0).sum(),
    "credit_limit_invalid": (df["credit_limit"] <= 0).sum(),
    "loan_gt_limit": (df["loan_amount"] > df["credit_limit"]).sum()
}

invalid_checks


{'age_invalid': 2000,
 'income_invalid': 1345,
 'credit_limit_invalid': 0,
 'loan_gt_limit': 29160}

In [75]:
cat_cols = df.select_dtypes(include=["object"]).columns

for col in cat_cols:
    print(f"\nCOLUMN: {col}")
    print(df[col].value_counts(dropna=False).head(15))



COLUMN: gender
gender
Female    16850
male      16772
Male      16736
M         16627
F         16542
female    16473
Name: count, dtype: int64

COLUMN: education_level
education_level
Graduate         37864
High School      28505
Post-Graduate    23994
NaN               5000
phd               1563
PhD               1557
Doctorate         1517
Name: count, dtype: int64

COLUMN: marital_status
marital_status
Married     47492
Single      38003
Divorced     9505
NaN          5000
Name: count, dtype: int64


In [76]:
# | Missing % | JPM Meaning   | Decision              |
# | --------- | ------------- | --------------------- |
# | 0–3%      | System noise  | Impute                |
# | 3–7%      | Business gaps | Impute + flag         |
# | 7–12%     | High risk     | Flag + careful impute |
# | >20%      | Unreliable    | Drop / redesign       |


In [77]:
# | Column             | Issue                  | Planned Action       |
# | ------------------ | ---------------------- | -------------------- |
# | annual_income      | 9.5% missing, outliers | Flag + median impute |
# | employment_years   | Missing                | Median impute        |
# | credit_utilization | Missing, extreme       | Cap + flag           |
# | age                | Missing                | Median impute        |
# | education_level    | Missing, dirty         | Standardize + mode   |
# | marital_status     | Missing                | Mode                 |
# | payment columns    | Missing / noise        | Minimal fix          |
# | customer_id        | Clean                  | Ignore               |
# | default_flag       | Clean                  | No touch             |


In [78]:
df.loc[(df["age"] < 18) | (df["age"] > 100), "age"] = np.nan
df["age"]=df["age"].fillna(df["age"].median())

In [79]:
df["annual_income"].skew()


14.336068566573763

In [80]:
df["annual_income"].quantile([0.5, 0.75, 0.9, 0.99])


0.50    4.468384e+05
0.75    6.427102e+05
0.90    9.174354e+05
0.99    4.542714e+06
Name: annual_income, dtype: float64

In [81]:
df.loc[df["annual_income"] <= 0, "annual_income"] = np.nan

In [82]:
df["annual_income_missing_flag"] = df["annual_income"].isna().astype(int)

df["annual_income"] = df["annual_income"].fillna(df["annual_income"].median())

In [83]:
df.loc[df["employment_years"] < 0, "employment_years"] = np.nan
df["employment_years"] = df["employment_years"].fillna(df["employment_years"].median())


In [84]:
df.loc[df["credit_limit"] <=0,"credit_limit"] = np.nan
df["credit_limit"] = df["credit_limit"].fillna(df["credit_limit"].median())

In [85]:
df.loc[df["loan_amount"] <= 0, "loan_amount"] = np.nan
df["loan_amount"] = df["loan_amount"].fillna(df["loan_amount"].median())
df.loc[df["loan_amount"] > df["credit_limit"], "loan_amount"] = df["credit_limit"]


In [86]:
df["credit_utilization"].describe()

count    95000.000000
mean         0.599817
std          0.288456
min          0.100003
25%          0.349955
50%          0.600458
75%          0.848543
max          1.099983
Name: credit_utilization, dtype: float64

In [87]:
df.loc[df["credit_utilization"] < 0, "credit_utilization"] = np.nan
df["credit_utilization_missing_flag"] = df["credit_utilization"].isna().astype(int)

df["credit_utilization"] = df["credit_utilization"].clip(upper=1.5)

df["credit_utilization"] = df["credit_utilization"].fillna(df["credit_utilization"].median())


In [88]:
cat_cols = ["gender", "education_level", "marital_status"]

for col in cat_cols:
    print(f"\nCOLUMN: {col}")
    print(df[col].value_counts(dropna=False))



COLUMN: gender
gender
Female    16850
male      16772
Male      16736
M         16627
F         16542
female    16473
Name: count, dtype: int64

COLUMN: education_level
education_level
Graduate         37864
High School      28505
Post-Graduate    23994
NaN               5000
phd               1563
PhD               1557
Doctorate         1517
Name: count, dtype: int64

COLUMN: marital_status
marital_status
Married     47492
Single      38003
Divorced     9505
NaN          5000
Name: count, dtype: int64


In [89]:
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

In [90]:
df["gender"] = df["gender"].replace({
    "m": "male",
    "male ": "male",
    "f": "female",
    "female ": "female",
    "unknown": np.nan,
    "nan": np.nan
})

In [91]:
df["education_level"] = df["education_level"].replace({
    "graduate": "graduate",
    "post graduate": "post-graduate",
    "postgraduate": "post-graduate",
    "doctorate": "doctorate",
    "phd": "phd",
    "na": np.nan,
    "": np.nan,
    "nan": np.nan
})


In [93]:
valid_categories = {
    "gender": ["male", "female"],
    "education_level": ["high school", "graduate", "post-graduate", "phd"],
    "marital_status": ["single", "married", "divorced", "widowed"]
}

for col, valid_vals in valid_categories.items():
    df.loc[~df[col].isin(valid_vals), col] = np.nan


In [94]:
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


In [95]:
for col in cat_cols:
    print(f"\n{col.upper()} VALUES:")
    print(df[col].value_counts())



GENDER VALUES:
gender
male      50135
female    49865
Name: count, dtype: int64

EDUCATION_LEVEL VALUES:
education_level
graduate         44381
high school      28505
post-graduate    23994
phd               3120
Name: count, dtype: int64

MARITAL_STATUS VALUES:
marital_status
married     52492
single      38003
divorced     9505
Name: count, dtype: int64


In [96]:
delay_cols = [f"pay_delay_m{i}" for i in range(1, 7)]

for col in delay_cols:
    df.loc[df[col] < 0, col] = np.nan
    df[col] = df[col].clip(upper=6)
    df[col] = df[col].fillna(0)


In [97]:
bill_cols = [f"bill_amt_m{i}" for i in range(1, 7)]

for col in bill_cols:
    df.loc[df[col] < 0, col] = np.nan
    
    upper_cap = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=upper_cap)
    
    df[col] = df[col].fillna(df[col].median())


In [98]:
pay_cols = [f"pay_amt_m{i}" for i in range(1, 7)]

for col in pay_cols:
    df.loc[df[col] < 0, col] = np.nan
    
    upper_cap = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=upper_cap)
    
    df[col] = df[col].fillna(0)


In [99]:
df.loc[df["num_late_payments"] < 0, "num_late_payments"] = 0


In [100]:
df.loc[df["past_defaults"] < 0, "past_defaults"] = 0
df["past_defaults"] = df["past_defaults"].fillna(0)


In [101]:
df[
    delay_cols + bill_cols + pay_cols + ["num_late_payments", "past_defaults"]
].describe().T


,count,mean,std,min,25%,50%,75%,max
pay_delay_m1,100000.0,0.631080,1.220046,0.0,0.000000,0.000000,1.000000,6.000000
pay_delay_m2,100000.0,0.624120,1.213272,0.0,0.000000,0.000000,1.000000,6.000000
pay_delay_m3,100000.0,0.625930,1.212681,0.0,0.000000,0.000000,1.000000,6.000000
pay_delay_m4,100000.0,0.632530,1.217682,0.0,0.000000,0.000000,1.000000,6.000000
pay_delay_m5,100000.0,0.627310,1.214037,0.0,0.000000,0.000000,1.000000,6.000000
pay_delay_m6,100000.0,0.624630,1.210158,0.0,0.000000,0.000000,1.000000,6.000000
bill_amt_m1,100000.0,124704.145213,98169.886492,0.0,54402.989437,105382.564141,171870.861638,479593.697896
bill_amt_m2,100000.0,124839.055971,98324.606954,0.0,54255.430359,105571.689439,172187.198264,478575.025066
bill_amt_m3,100000.0,124848.113093,98261.976115,0.0,54568.682148,105411.504216,171994.563185,478842.221683
bill_amt_m4,100000.0,124652.555902,98478.226516,0.0,54239.651275,105065.201367,171641.333210,481120.442077


In [102]:
df.isna().sum().sort_values(ascending=False).head(10)

monthly_income    3000
customer_id       1000
pay_amt_m5           0
bill_amt_m3          0
pay_amt_m3           0
pay_delay_m4         0
bill_amt_m4          0
pay_amt_m4           0
pay_delay_m5         0
bill_amt_m5          0
dtype: int64

In [103]:
df["monthly_income"] = df["annual_income"] / 12

In [104]:
df["customer_id"] = df["customer_id"].fillna(method="ffill")

C:\Users\soumy\AppData\Local\Temp\ipykernel_20276\1403913278.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["customer_id"] = df["customer_id"].fillna(method="ffill")


In [106]:
df.isna().sum().sort_values(ascending=False).head(10)

customer_id     0
pay_amt_m2      0
bill_amt_m3     0
pay_amt_m3      0
pay_delay_m4    0
bill_amt_m4     0
pay_amt_m4      0
pay_delay_m5    0
bill_amt_m5     0
pay_amt_m5      0
dtype: int64

In [107]:
df.to_csv("data_processed_credit_data_cleaned_v2.csv", index=False)
